# Multi-Seed Ensemble — RoBERTa-base (Kaggle)

## Strategy
1. **Phase 1** — Train `roberta-base` on `train + aug_train` with **5 different seeds**, evaluate each on the dev split, keep the **top 3** by macro-F1.
2. **`dev.txt`** — Written from the single best Phase 1 model.
3. **Phase 2** — Retrain the top-3 seeds from scratch on the **full dataset** (`train + aug_train + dev + aug_dev`), no held-out validation.
4. **`test.txt`** — **Majority-vote ensemble** of the 3 Phase 2 models (odd number → no ties).

Each Phase 1/2 model is freed from GPU memory after use to stay within Kaggle's VRAM budget.

In [ ]:
!pip install -q contractions python-dotenv huggingface_hub

In [ ]:
import gc
import os
import re
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns
import contractions

from dotenv import load_dotenv
from huggingface_hub import login

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Optional: HF login (needed if roberta-base requires auth, or for private models)
load_dotenv()
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print('HF token loaded.')
else:
    print('No HF_TOKEN found; proceeding without login.')

In [ ]:
# ============================================================
# Configuration  ← edit paths to match your Kaggle dataset
# ============================================================
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 256
NUM_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3

# Seeds to try in Phase 1; top TOP_K are carried into Phase 2
SEEDS = [42, 123, 256, 512, 1024]
TOP_K = 3

DATA_SEED = 42   # fixed seed for data shuffling (independent of training seeds)

# ── Kaggle paths ─────────────────────────────────────────────────
DATA_ROOT = '/kaggle/input/pcl-dataset'   # folder containing dontpatronizeme_pcl.tsv etc.
CKPT_ROOT = '/kaggle/working/checkpoints'
# ─────────────────────────────────────────────────────────────────

TSV_PATH       = os.path.join(DATA_ROOT, 'dontpatronizeme_pcl.tsv')
TRAIN_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv')
DEV_IDS_PATH   = os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv')
TEST_PATH      = os.path.join(DATA_ROOT, 'test', 'task4_test.tsv')
AUG_TRAIN_PATH = os.path.join(DATA_ROOT, 'data_augmentation', 'augmented_train_data.csv')
AUG_DEV_PATH   = os.path.join(DATA_ROOT, 'data_augmentation', 'augmented_dev_data.csv')

DEV_TXT_PATH  = '/kaggle/working/dev.txt'
TEST_TXT_PATH = '/kaggle/working/test.txt'

os.makedirs(CKPT_ROOT, exist_ok=True)
print('Config OK.')

In [ ]:
# ============================================================
# Helper functions
# ============================================================
def load_task1(tsv_path: str) -> pd.DataFrame:
    rows = []
    with open(tsv_path, encoding='utf-8') as f:
        for line in f.readlines()[4:]:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue
            orig_label = parts[-1]
            rows.append({
                'par_id':  str(parts[0]),
                'art_id':  parts[1],
                'keyword': parts[2],
                'country': parts[3],
                'text':    parts[4],
                'label':   0 if orig_label in {'0', '1'} else 1,
            })
    return pd.DataFrame(rows)


def load_test_tsv(test_path: str) -> pd.DataFrame:
    rows = []
    with open(test_path, encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 5:
                continue
            rows.append({
                'par_id':  parts[0],
                'art_id':  parts[1],
                'keyword': parts[2],
                'country': parts[3],
                'text':    parts[4],
            })
    return pd.DataFrame(rows)


def preprocess_text(text: str) -> str:
    text = str(text)
    text = contractions.fix(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def make_model_text(frame: pd.DataFrame) -> pd.Series:
    kw  = frame['keyword'].fillna('').astype(str).str.strip()
    cc  = frame['country'].fillna('').astype(str).str.strip()
    txt = frame['clean_text'].fillna('').astype(str).str.strip()
    return kw + ' </s> ' + cc + ' </s> ' + txt


def load_aug_csv(path: str) -> pd.DataFrame:
    aug = pd.read_csv(path, dtype={'par_id': str})
    aug = aug[aug['label'].astype(str) != 'label'].copy()
    aug['label'] = aug['label'].astype(int)
    if 'clean_text' in aug.columns:
        aug['clean_text'] = aug['clean_text'].fillna(aug['text'].apply(preprocess_text))
    else:
        aug['clean_text'] = aug['text'].apply(preprocess_text)
    aug['model_text'] = make_model_text(aug)
    return aug[['par_id', 'keyword', 'country', 'text', 'clean_text', 'model_text', 'label']]

In [ ]:
# ============================================================
# Load + preprocess labelled data
# ============================================================
df = load_task1(TSV_PATH)
df['clean_text'] = df['text'].apply(preprocess_text)
df['model_text'] = make_model_text(df)

print(f'Full dataset: {len(df):,} rows')
print(df['label'].value_counts().rename({0: 'No-PCL', 1: 'PCL'}))

In [ ]:
# ============================================================
# Official train / dev split + augmented data
# ============================================================
train_ids_df = pd.read_csv(TRAIN_IDS_PATH, dtype={'par_id': str})
dev_ids_df   = pd.read_csv(DEV_IDS_PATH,   dtype={'par_id': str})

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids   = set(dev_ids_df['par_id'].astype(str))

train_base_df = df[df['par_id'].isin(train_par_ids)].copy().reset_index(drop=True)

# Dev ordered exactly as official split file (required for correct dev.txt)
dev_base_df = (
    dev_ids_df[['par_id']]
    .merge(df[df['par_id'].isin(dev_par_ids)], on='par_id', how='left')
    .reset_index(drop=True)
)

leftover = df[~df['par_id'].isin(train_par_ids | dev_par_ids)]
if len(leftover):
    train_base_df = pd.concat([train_base_df, leftover], ignore_index=True)
    print(f'Appended {len(leftover):,} leftover samples to train.')

aug_train_df = load_aug_csv(AUG_TRAIN_PATH)
aug_dev_df   = load_aug_csv(AUG_DEV_PATH)

print(f'train_base : {len(train_base_df):,} | PCL={int((train_base_df.label==1).sum()):,}')
print(f'dev_base   : {len(dev_base_df):,}  | PCL={int((dev_base_df.label==1).sum()):,}')
print(f'aug_train  : {len(aug_train_df):,} | PCL={int((aug_train_df.label==1).sum()):,}')
print(f'aug_dev    : {len(aug_dev_df):,}   | PCL={int((aug_dev_df.label==1).sum()):,}')

In [ ]:
# ============================================================
# Tokeniser + dataset builder
# ============================================================
tokenizer     = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)


def build_dataset(frame: pd.DataFrame, has_labels: bool = True) -> Dataset:
    cols = ['model_text', 'label'] if has_labels else ['model_text']
    ds = (
        Dataset.from_pandas(
            frame[cols].rename(columns={'model_text': 'text'}),
            preserve_index=False,
        )
        .map(tokenize, batched=True, remove_columns=['text'])
    )
    if has_labels:
        ds = ds.rename_column('label', 'labels')
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    else:
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    return ds

In [ ]:
# ============================================================
# Metrics + custom weighted-CE Trainer
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "f1_pcl": f1_score(labels, preds, average="binary", pos_label=1, zero_division=0),
        "precision_pcl": precision_score(labels, preds, average="binary", pos_label=1, zero_division=0),
        "recall_pcl": recall_score(labels, preds, average="binary", pos_label=1, zero_division=0),
    }



class WeightedCETrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = (
            torch.tensor(class_weights, dtype=torch.float)
            if class_weights is not None else None
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        weight  = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss    = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

---
## Phase 1 — Multi-seed training on train + aug_train
Each seed produces an independent model. The top `TOP_K` by dev macro-F1 advance to Phase 2.

In [ ]:
# ============================================================
# Build Phase 1 dataset  (done once; same data for all seeds)
# ============================================================
p1_train_df = (
    pd.concat([train_base_df, aug_train_df], ignore_index=True)
    .sample(frac=1, random_state=DATA_SEED)
    .reset_index(drop=True)
)

p1_cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=p1_train_df['label'].values)

print(f'Phase 1 train: {len(p1_train_df):,} | PCL={int((p1_train_df.label==1).sum()):,} | No-PCL={int((p1_train_df.label==0).sum()):,}')
print(f'Phase 1 dev  : {len(dev_base_df):,}')
print(f'Class weights -> No-PCL: {p1_cw[0]:.4f},  PCL: {p1_cw[1]:.4f}')

p1_train_ds = build_dataset(p1_train_df)
p1_dev_ds   = build_dataset(dev_base_df)

In [ ]:
# ============================================================
# Phase 1 multi-seed training loop
# ============================================================
p1_results = []   # list of dicts: seed, f1_macro, best_dir, dev_preds

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'  Phase 1  |  Seed {seed}')
    print(f'{"="*60}')

    set_seed(seed)

    ckpt_dir = os.path.join(CKPT_ROOT, f'phase1_seed{seed}')
    os.makedirs(ckpt_dir, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    args = TrainingArguments(
        output_dir=ckpt_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        optim='adamw_torch',
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_pcl',
        greater_is_better=True,
        save_total_limit=2,
        report_to='none',
        seed=seed,
    )

    trainer = WeightedCETrainer(
        model=model,
        args=args,
        train_dataset=p1_train_ds,
        eval_dataset=p1_dev_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=p1_cw.tolist(),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    trainer.train()

    # Predict on dev with the best checkpoint (already loaded back)
    pred_out  = trainer.predict(p1_dev_ds)
    dev_preds = np.argmax(pred_out.predictions, axis=-1)
    f1        = f1_score(pred_out.label_ids, dev_preds, average='macro', zero_division=0)

    # Persist best model so we can reload it later for ensembling
    best_dir = os.path.join(ckpt_dir, 'best')
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    p1_results.append({'seed': seed, 'f1_macro': f1, 'best_dir': best_dir, 'dev_preds': dev_preds})
    print(f'  Seed {seed} → dev macro-F1: {f1:.4f}')

    # Free GPU memory before next seed
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# ============================================================
# Phase 1 results — select top TOP_K seeds
# ============================================================
p1_results.sort(key=lambda x: x['f1_macro'], reverse=True)
top_results = p1_results[:TOP_K]
top_seeds   = [r['seed'] for r in top_results]

print('Phase 1 leaderboard:')
print(f'{"Seed":>6}  {"macro-F1":>10}  {"Selected"}')
print('-' * 32)
for r in p1_results:
    flag = '✓' if r['seed'] in top_seeds else ' '
    print(f'{r["seed"]:>6}  {r["f1_macro"]:>10.4f}  {flag}')

print(f'\nTop {TOP_K} seeds advancing to Phase 2: {top_seeds}')

In [ ]:
# ============================================================
# Evaluation report for the best Phase 1 model
# ============================================================
best_p1 = p1_results[0]   # sorted descending, so index 0 is the best
y_true  = p1_dev_ds['labels']
y_pred  = best_p1['dev_preds']

print(f'Best Phase 1 model  seed={best_p1["seed"]}  macro-F1={best_p1["f1_macro"]:.4f}\n')
print(classification_report(y_true, y_pred, target_names=['No-PCL', 'PCL'], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred No-PCL', 'Pred PCL'],
            yticklabels=['True No-PCL', 'True PCL'])
plt.title(f'Confusion Matrix — Best Phase 1 (seed={best_p1["seed"]})')
plt.tight_layout()
plt.savefig(os.path.join(CKPT_ROOT, 'phase1_best_cm.png'), dpi=150)
plt.show()

In [ ]:
# ============================================================
# Generate dev.txt  (best Phase 1 model, official order)
# ============================================================
dev_preds_list = best_p1['dev_preds'].tolist()

assert set(dev_preds_list).issubset({0, 1})
assert len(dev_preds_list) == len(dev_base_df)

with open(DEV_TXT_PATH, 'w') as f:
    f.write('\n'.join(str(p) for p in dev_preds_list) + '\n')

print(f'Written {len(dev_preds_list):,} predictions -> {DEV_TXT_PATH}')

---
## Phase 2 — Full retrain with top-3 seeds
Each seed is trained from scratch on `train + aug_train + dev + aug_dev`.
No held-out validation; each model is freed from GPU memory after saving.

In [ ]:
# ============================================================
# Build Phase 2 dataset  (done once; same data for all seeds)
# ============================================================
p2_train_df = (
    pd.concat([train_base_df, aug_train_df, dev_base_df, aug_dev_df], ignore_index=True)
    .sample(frac=1, random_state=DATA_SEED)
    .reset_index(drop=True)
)

p2_cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=p2_train_df['label'].values)

print(f'Phase 2 train: {len(p2_train_df):,} | PCL={int((p2_train_df.label==1).sum()):,} | No-PCL={int((p2_train_df.label==0).sum()):,}')
print(f'Class weights -> No-PCL: {p2_cw[0]:.4f},  PCL: {p2_cw[1]:.4f}')

p2_train_ds = build_dataset(p2_train_df)

In [ ]:
# ============================================================
# Phase 2 training loop  (top-3 seeds)
# ============================================================
p2_model_dirs = []   # final saved model paths, in top-seed order

for r in top_results:
    seed = r['seed']
    print(f'\n{"="*60}')
    print(f'  Phase 2  |  Seed {seed}')
    print(f'{"="*60}')

    set_seed(seed)

    ckpt_dir = os.path.join(CKPT_ROOT, f'phase2_seed{seed}')
    os.makedirs(ckpt_dir, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    args = TrainingArguments(
        output_dir=ckpt_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        optim='adamw_torch',
        eval_strategy='no',
        save_strategy='no',
        report_to='none',
        seed=seed,
    )

    trainer = WeightedCETrainer(
        model=model,
        args=args,
        train_dataset=p2_train_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        class_weights=p2_cw.tolist(),
    )

    trainer.train()

    final_dir = os.path.join(ckpt_dir, 'final')
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)
    p2_model_dirs.append(final_dir)
    print(f'  Saved Phase 2 model (seed={seed}) -> {final_dir}')

    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

---
## Ensemble — Majority voting on test set
Each Phase 2 model predicts in turn; its predictions are accumulated and freed before the next is loaded.
With 3 models, majority = at least 2 voting for the same label.

In [ ]:
# ============================================================
# Load + tokenise test set
# ============================================================
test_df = load_test_tsv(TEST_PATH)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)
test_df['model_text'] = make_model_text(test_df)

test_ds = build_dataset(test_df, has_labels=False)
print(f'Test samples: {len(test_df):,}  (expected 3832)')

In [ ]:
# ============================================================
# Collect predictions from each Phase 2 model
# ============================================================
def predict_labels(model, dataset, collator, batch_size=32):
    """Run inference on dataset; return integer predictions as a numpy array."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator)
    preds  = []
    with torch.no_grad():
        for batch in loader:
            batch   = {k: v.to(device) for k, v in batch.items()}
            logits  = model(**batch).logits
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
    return np.array(preds, dtype=int)


all_test_preds = []   # shape after loop: (TOP_K, n_test)

for i, (model_dir, r) in enumerate(zip(p2_model_dirs, top_results)):
    print(f'Predicting with model {i+1}/{TOP_K}  (seed={r["seed"]}) ...')
    p2_model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    preds    = predict_labels(p2_model, test_ds, data_collator)
    all_test_preds.append(preds)
    print(f'  PCL rate: {preds.mean():.3f}')

    del p2_model
    torch.cuda.empty_cache()
    gc.collect()

# Majority vote: stack → (n_test, TOP_K) → sum → >= ceil(TOP_K/2)
votes          = np.stack(all_test_preds, axis=1)          # (n_test, 3)
threshold      = (TOP_K // 2) + 1                          # 2 for TOP_K=3
ensemble_preds = (votes.sum(axis=1) >= threshold).astype(int).tolist()

print(f'\nEnsemble PCL rate : {np.mean(ensemble_preds):.3f}')
print(f'Ensemble 0s / 1s  : {ensemble_preds.count(0):,} / {ensemble_preds.count(1):,}')

In [ ]:
# ============================================================
# Generate test.txt
# ============================================================
assert set(ensemble_preds).issubset({0, 1})
assert len(ensemble_preds) == len(test_df), (
    f'Test count mismatch: {len(ensemble_preds)} vs {len(test_df)}'
)

with open(TEST_TXT_PATH, 'w') as f:
    f.write('\n'.join(str(p) for p in ensemble_preds) + '\n')

print(f'Written {len(ensemble_preds):,} predictions -> {TEST_TXT_PATH}  (expected 3832)')

In [ ]:
# ============================================================
# Sanity checks
# ============================================================
for path in [DEV_TXT_PATH, TEST_TXT_PATH]:
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]
    print(f'{os.path.basename(path):12s}: lines={len(lines):,}, unique_labels={sorted(set(lines))}')